#### Pipeline Metrics for the End to End pipeline job execution

#### 1. Cost and time for the  task to execute

In [0]:
%sql
    
WITH task_deduped AS (
  -- Deduplicate timeline slices: one row per task with actual start → end
  SELECT
    t.job_id,
    t.job_run_id,
    t.task_key,
    MIN(t.period_start_time)                          AS task_start,
    MAX(t.period_end_time)                            AS task_end,
    TIMESTAMPDIFF(SECOND, MIN(t.period_start_time), MAX(t.period_end_time)) AS duration_sec,
    MAX(t.setup_duration_seconds)                      AS setup_sec,
    MAX(t.execution_duration_seconds)                  AS exec_sec
  FROM system.lakeflow.job_task_run_timeline t
  WHERE t.job_id = '398813427807450'
    AND t.job_run_id = '26336279038509'
  GROUP BY t.job_id, t.job_run_id, t.task_key
),
job_summary AS (
  SELECT
    job_id,
    job_run_id,
    COUNT(*)                                          AS total_tasks,
    MIN(task_start)                                    AS job_start,
    MAX(task_end)                                      AS job_end,
    ROUND(TIMESTAMPDIFF(SECOND, MIN(task_start), MAX(task_end)) / 60.0, 2) AS wall_clock_min,
    ROUND(SUM(duration_sec) / 60.0, 2)                AS total_task_min,
    ROUND(SUM(COALESCE(setup_sec, 0)) / 60.0, 2)      AS total_setup_min,
    ROUND(SUM(COALESCE(exec_sec, 0)) / 60.0, 2)       AS total_exec_min
  FROM task_deduped
  GROUP BY job_id, job_run_id
),
job_name AS (
  SELECT job_id, run_id AS job_run_id, run_name
  FROM system.lakeflow.job_run_timeline
  WHERE job_id = '398813427807450'
    AND run_id = '26336279038509'
  LIMIT 1
),
job_cost AS (
  SELECT
    u.usage_metadata.job_id,
    u.usage_metadata.job_run_id,
    SUM(u.usage_quantity)                              AS total_dbus,
    ROUND(SUM(u.usage_quantity * p.pricing.default), 4) AS total_cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices p
    ON u.cloud = p.cloud
    AND u.sku_name = p.sku_name
    AND u.usage_start_time >= p.price_start_time
    AND (u.usage_end_time <= p.price_end_time OR p.price_end_time IS NULL)
  WHERE u.usage_metadata.job_id = '398813427807450'
    AND u.usage_metadata.job_run_id = '26336279038509'
  GROUP BY ALL
)
SELECT
  COALESCE(n.run_name, 'Job ' || s.job_id)            AS job_name,
  s.total_tasks,
  s.job_start,
  s.job_end,
  s.wall_clock_min,
  s.total_task_min,
  s.total_exec_min,
  s.total_setup_min,
  c.total_dbus,
  COALESCE(c.total_cost_usd, 0)                       AS total_cost_usd,
  CASE WHEN c.total_cost_usd IS NULL THEN 'Billing data not yet available (1-2 day delay)'
       ELSE 'Available' END                            AS cost_status
FROM job_summary s
LEFT JOIN job_name n ON s.job_id = n.job_id AND s.job_run_id = n.job_run_id
LEFT JOIN job_cost c ON s.job_id = c.job_id AND s.job_run_id = c.job_run_id;

#### 2. Cost and time for Classic All-Purpose Job Cluster

In [0]:
%sql
    
    
WITH task_deduped AS (
  -- Deduplicate timeline slices: one row per task with actual start → end
  SELECT
    t.job_id,
    t.job_run_id,
    t.task_key,
    MIN(t.period_start_time)                          AS task_start,
    MAX(t.period_end_time)                            AS task_end,
    TIMESTAMPDIFF(SECOND, MIN(t.period_start_time), MAX(t.period_end_time)) AS duration_sec,
    MAX(t.setup_duration_seconds)                      AS setup_sec,
    MAX(t.execution_duration_seconds)                  AS exec_sec
  FROM system.lakeflow.job_task_run_timeline t
  WHERE t.job_id = '1049721677717945'
    AND t.job_run_id = '733462842215316'
  GROUP BY t.job_id, t.job_run_id, t.task_key
),
job_summary AS (
  SELECT
    job_id,
    job_run_id,
    COUNT(*)                                          AS total_tasks,
    MIN(task_start)                                    AS job_start,
    MAX(task_end)                                      AS job_end,
    ROUND(TIMESTAMPDIFF(SECOND, MIN(task_start), MAX(task_end)) / 60.0, 2) AS wall_clock_min,
    ROUND(SUM(duration_sec) / 60.0, 2)                AS total_task_min,
    ROUND(SUM(COALESCE(setup_sec, 0)) / 60.0, 2)      AS total_setup_min,
    ROUND(SUM(COALESCE(exec_sec, 0)) / 60.0, 2)       AS total_exec_min
  FROM task_deduped
  GROUP BY job_id, job_run_id
),
job_name AS (
  SELECT job_id, run_id AS job_run_id, run_name
  FROM system.lakeflow.job_run_timeline
  WHERE job_id = '1049721677717945'
    AND run_id = '733462842215316'
  LIMIT 1
),
job_workspace AS (
  SELECT workspace_id
  FROM system.lakeflow.job_run_timeline
  WHERE job_id = '1049721677717945'
    AND run_id = '733462842215316'
  LIMIT 1
),
-- Tier 1: Direct job-level billing (serverless / dedicated job clusters)
job_cost_direct AS (
  SELECT
    u.usage_metadata.job_id,
    u.usage_metadata.job_run_id,
    SUM(u.usage_quantity)                              AS total_dbus,
    ROUND(SUM(u.usage_quantity * p.pricing.default), 4) AS total_cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices p
    ON u.cloud = p.cloud
    AND u.sku_name = p.sku_name
    AND u.usage_start_time >= p.price_start_time
    AND (u.usage_end_time <= p.price_end_time OR p.price_end_time IS NULL)
  WHERE u.usage_metadata.job_id = '1049721677717945'
    AND u.usage_metadata.job_run_id = '733462842215316'
  GROUP BY ALL
),
-- Tier 2: Cluster-level billing with time-proportional estimation
-- Used when job ran on a classic all-purpose cluster (billing attributed to cluster, not job)
cluster_cost AS (
  SELECT
    ROUND(SUM(
      u.usage_quantity *
      GREATEST(0, TIMESTAMPDIFF(SECOND,
        GREATEST(u.usage_start_time, s.job_start),
        LEAST(u.usage_end_time, s.job_end)
      )) * 1.0 /
      NULLIF(TIMESTAMPDIFF(SECOND, u.usage_start_time, u.usage_end_time), 0)
    ), 6) AS est_dbus,
    ROUND(SUM(
      u.usage_quantity * p.pricing.default *
      GREATEST(0, TIMESTAMPDIFF(SECOND,
        GREATEST(u.usage_start_time, s.job_start),
        LEAST(u.usage_end_time, s.job_end)
      )) * 1.0 /
      NULLIF(TIMESTAMPDIFF(SECOND, u.usage_start_time, u.usage_end_time), 0)
    ), 4) AS est_cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices p
    ON u.cloud = p.cloud
    AND u.sku_name = p.sku_name
    AND u.usage_start_time >= p.price_start_time
    AND (u.usage_end_time <= p.price_end_time OR p.price_end_time IS NULL)
  CROSS JOIN job_summary s
  CROSS JOIN job_workspace w
  WHERE u.usage_metadata.cluster_id IS NOT NULL
    AND u.usage_metadata.job_id IS NULL
    AND u.workspace_id = w.workspace_id
    AND u.billing_origin_product = 'ALL_PURPOSE'
    AND u.usage_date BETWEEN DATE(s.job_start) AND DATE(s.job_end)
    AND u.usage_start_time < s.job_end
    AND u.usage_end_time > s.job_start
)
SELECT
  COALESCE(n.run_name, 'Job ' || s.job_id)            AS job_name,
  s.total_tasks,
  s.job_start,
  s.job_end,
  s.wall_clock_min,
  s.total_task_min,
  s.total_exec_min,
  s.total_setup_min,
  COALESCE(jc.total_dbus, cc.est_dbus)                AS total_dbus,
  COALESCE(jc.total_cost_usd, cc.est_cost_usd, 0)    AS total_cost_usd,
  CASE
    WHEN jc.total_cost_usd IS NOT NULL THEN 'Exact (job-level billing)'
    WHEN cc.est_cost_usd IS NOT NULL THEN 'Estimated (all-purpose cluster, time pro-rated)'
    ELSE 'Billing data not yet available (1-2 day delay)'
  END                                                  AS cost_status
FROM job_summary s
LEFT JOIN job_name n ON s.job_id = n.job_id AND s.job_run_id = n.job_run_id
LEFT JOIN job_cost_direct jc ON s.job_id = jc.job_id AND s.job_run_id = jc.job_run_id
LEFT JOIN cluster_cost cc ON 1=1;